### TODO:
COMENZAR YA CON EL MODELO DE DIFUSUION PARA AUDIO, PERO ANTES ARREGLAR LO DE LAS CAPAS DEL MODELO

In [ ]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import time

import torchaudio.transforms as T
import math   
from src.dataset import LatentNSynth 
import torch
from src.diffusion import *
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae_model_path = r"C:\Users\Articuno\Desktop\TFG-info\data\models\vae.pth"

model_path = r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\mod_stable_diff_music.pth"
sch_path =  r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\sch_stable_diff_music.pth"
temp_path = r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\temp_stable_diff_music.pth"
sch_temp_path = r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\temp_sch_stable_diff_music.pth" 

# Pasos:
Cargar el VAE entrenado

-> Lo mismo es más rápido guardar el espacio latente del VAE para cada audio y cargarlo directamente, en vez de cargar el modelo y pasar cada audio por el encoder

-> El vae recibe wavs o espectrogramas?

Entrenar el modelo de difusión con el espacio latente del VAE
    ¿Evaluamos con el espacio latente o con el audio reconstruido? -> No reconstruimos el audio, tardariamos mucho y arrastrariamos el error del vae

In [ ]:
# TODO 
@torch.no_grad()
def sample_images(model, scheduler, num_images=4, image_size=(1, 28, 28)):
    model.eval()

    x = torch.randn(num_images, *image_size, device=device)
    T = scheduler.beta.size(0)

    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar

    for t in reversed(range(T)):
        t_batch = torch.full((num_images,), t, device=device, dtype=torch.long)

        # predicción de ruido
        e_pred = model(x, t_batch)
        
        beta_t = betas[t]
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]
        
        # beta_t = beta_t.view(1,1,1,1)
        # alpha_t = alpha_t.view(1,1,1,1)
        # alpha_bar_t = alpha_bar_t.view(1,1,1,1)


        # Coeficientes DDPM
        coef1 = 1 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)

        mu = coef1 * (x - coef2 * e_pred)

        if t > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t)
            x = mu + sigma_t * noise
        else:
            x = mu

    x = x.clamp(0, 1).cpu()

    # Mostrar imágenes
    fig, axes = plt.subplots(1, num_images, figsize=(num_images*2, 2))
    for i in range(num_images):
        axes[i].imshow(x[i, 0], cmap='gray')
        axes[i].axis('off')
    plt.show()

    return x

@torch.no_grad()
def remove_noise(model, scheduler, image_size=(1, 28, 28), x=None, alpha=0):
    model.eval()
    if x is None:
        x = torch.randn(1, *image_size, device=device)
        alpha = 1

    # Guardar imagen original
    x_original = x.clone().cpu()

    # Añadir ruido
    noise = torch.randn(1, *image_size, device=device) * (1 - alpha)
    x_noisy = x * alpha + noise
    x = x_noisy.clone()  # este será el tensor que pasaremos al DDPM

    T = scheduler.beta.size(0)
    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar

    for t in reversed(range(T)):
        t_batch = torch.full((1,), t, device=device, dtype=torch.long)
        e_pred = model(x, t_batch)
        
        beta_t = betas[t]
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]

        coef1 = 1 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)
        mu = coef1 * (x - coef2 * e_pred)

        if t > 0:
            noise_step = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t)
            x = mu + sigma_t * noise_step
        else:
            x = mu

    x_denoised = x.clamp(0, 1).cpu()

    # Mostrar imágenes: original, con ruido, denoised
    fig, axes = plt.subplots(1, 3, figsize=(6, 2))
    axes[0].imshow(x_original[0, 0], cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(x_noisy[0, 0].cpu(), cmap='gray')
    axes[1].set_title('Noisy')
    axes[1].axis('off')

    axes[2].imshow(x_denoised[0, 0], cmap='gray')
    axes[2].set_title('Denoised')
    axes[2].axis('off')

    plt.show()

    return x_denoised

In [ ]:
from src.utils.dataset import *
from src.utils.audio_utils import *
from torch.utils.data import Dataset

class LatentNSynth(Dataset):
    def __init__(self, nsynth, vae_model_path, stft_transform):
        super().__init__()
        self.nsynth = nsynth
        self.stft_transform = stft_transform

        # Cargar el modelo VAE
        self.vae = torch.load(vae_model_path, map_location=device)
        self.vae.eval()

    def __len__(self):
        return len(self._keys)

    def __getitem__(self, index):
        waveform, sample_rate, key, metadata = self.nsynth[index]
        # Convertir a mono si es estéreo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        # Normalizar el audio
        waveform = waveform / torch.max(torch.abs(waveform))
        # Calcular el espectrograma
        spectrogram = self.stft_transform(waveform)
        # Obtener magnitud y fase
        # mag, sin, cos = compute_magnitude_and_phase_sin_cos(spectrogram) ## SINCOS
        
        # Usar el vae para obtener la imagen latente
        with torch.no_grad():
            latent = self.vae.encode(spectrogram.to(device)) # TODO no estoy seguro de que sea así
        return latent, sample_rate, key, metadata # TODO tampoco estoy seguro de si tengo que devolver el sample_rate, key y metadata, o si solo me interesa el latent para entrenar el modelo de difusión

In [ ]:
def precalculate_latent_dataset():
    ''' TODO, codifica el dataset y lo guarda en el disco'''
    pass 

In [ ]:
def train(
    model, 
    stft_transform,
    epochs=100, 
    batch_size=32, 
    lr=1e-3, 
    model_path=None,
    sch_path=None, 
    dataset=LatentNSynth('training', None), 
    verb=True,
    verb_batch=False
):
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    scheduler = Scheduler(num_epochs=epochs, device=device).to(device)
    diffuser = Diffuser(model, scheduler).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)    
    scaler = torch.cuda.amp.GradScaler()
    
    mse_loss = nn.MSELoss()
    best_loss = 1
    losses = []
        
    for epoch in range(epochs):
        start_time = time.time() 
        b_idx = 0
        epoch_loss = 0
        for wave, _, _, _ in train_loader:
            b_idx += 1
            # wave = wave.to(device)
            # x = stft_transform(wave)
            wave = wave.to(device)
            stft_spec = stft_transform(wave)
            # print(f"stft_spec: {stft_spec.shape}")
            # mag, phase = compute_magnitude_and_phase(stft_spec) ## SINCOS
            mag, sin, cos = compute_magnitude_and_phase_sin_cos(stft_spec) ## SINCOS
            log_mag = torch.log1p(mag) 
            # x = torch.cat([log_mag, phase], dim=1).to(device)  ## SINCOS
            x = torch.cat([log_mag, sin, cos], dim=1).to(device)  ## SINCOS

            batch_size = x.size(0)
            t = torch.randint(0, epochs, (batch_size,), device=device, dtype=torch.long)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(dtype=torch.bfloat16): # OPTIMIZACION FLOAT 32
                # print(f"mag: {mag.shape}, sin: {sin.shape}, cos: {cos.shape}, x: {x.shape}")
                z, e = diffuser(x, t)
                e_pred = model(z, t)
                if e_pred.shape != e.shape: # TODO borrar esto es debug
                    print('error en los tamaños')

                loss = mse_loss(e_pred, e)
                epoch_loss += loss.item()


            scaler.scale(loss).backward() # TODO no se si la funcion de loss es la mas optima para el caso
            scaler.step(optimizer)
            scaler.update()
            
            if verb_batch:
                if b_idx %10 == 0:
                    _t = time.time() - start_time
                    print(f'tiempo en {b_idx} batchs: {_t:.5f} \t| media de tiempo por batch: {_t/b_idx:.5f} \t| media de loss: {epoch_loss/b_idx:.5f}')

        losses.append(epoch_loss/b_idx)
            
        if verb:
            _t = time.time() - start_time
            print(f"Epoch {epoch}, Loss: {loss.item()}, time: {time.time() - start_time}, avg wave time: {_t/(batch_size*b_idx)}")
            
        if loss.item() < best_loss:
            best_loss = loss.item()
            if model_path is not None:
                torch.save(model.state_dict(), model_path)
                torch.save(scheduler.state_dict(), sch_path)
                if verb:
                    print(f"Model temporally saved at epoch {epoch} with loss {best_loss}")
            else:
                torch.save(model.state_dict(), temp_path)
                torch.save(scheduler.state_dict(), sch_temp_path)
                if verb:
                    print(f"Model saved at epoch {epoch} with loss {best_loss}")
    
    if model_path is not None:
        model.load_state_dict(torch.load(model_path))
        scheduler.load_state_dict(torch.load(sch_path))
    else:
        model.load_state_dict(torch.load(temp_path))
        scheduler.load_state_dict(torch.load(sch_temp_path))

        # eliminar temp?
        
    print("Training completed., best loss:", best_loss)
    
    return model, scheduler, losses
    

In [ ]:
# STFT transform
sample_rate = 16000
n_fft = 1500 # DISMINUIR TAMAÑO PARA OPTIMIZAR
hop_length = 250
win_length = n_fft
stft_transform = T.Spectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length,
    power=None, onesided=False, center=False
).to(device)
istft_transform = T.InverseSpectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length, onesided=False
).to(device)

# TRAIN SETUP
batch_size = 32
learning_rate = 1e-4
epochs = 20
# train_loader = DataLoader(NSynth('training'), batch_size=batch_size, shuffle=True, pin_memory=True)
# valid_loader = DataLoader(NSynth('validation'), batch_size=batch_size, shuffle=True, pin_memory=True)

# MODEL SETUP
input_height = win_length
input_width = 251 # por que es hop_length +1? 
input_size = (input_height, input_width)
emb_dim = 128
norm_groups = 6


# layers
sin = sout = c = 12  # canal base, reducido para no explotar memoria
# TODO hacer que el tamaño dependa de n_ftt
down_layers = [  
    DummyLayer(c,    c*2,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c canales
    DummyLayer(c*2,  c*4,  norm_groups, emb_dim, skip=True, stride=1).to(device),  # skip: c*2 canales
]

bottleneck = DummyLayer(c*4, c*4, norm_groups, emb_dim).to(device)

up_layers = [ # TODO Podria hacer esto con stride = -2? o coger y hacer que si el stride es negativo, aumente manualmente 
    DummyLayer(c*4 + c*4, c*2, norm_groups, emb_dim, stride=1).to(device),
    DummyLayer(c*2 + c*2, c, norm_groups, emb_dim, stride=1).to(device),
]

# up_layers = [
#     DummyLayer(c*16 + c*8, c*8, norm_groups, emb_dim).to(device),
#     DummyLayer(c*8  + c*4, c*4, norm_groups, emb_dim).to(device),
#     DummyLayer(c*4  + c*2, c*2, norm_groups, emb_dim).to(device),
#     DummyLayer(c*2  + c,   c,   norm_groups, emb_dim).to(device),
# ]

# embeder
embedder = Embeder(num_epochs=epochs, embed_dim=emb_dim, device=device).to(device)

# scheduler
scheduler = Scheduler(num_epochs=epochs, device=device).to(device)

# model
model = DiffusionModel(
    layer_channels=(sin, sout),
    norm_groups=norm_groups,
    up_layers=up_layers,
    down_layers=down_layers,
    bottleneck=bottleneck,
    embedder=embedder, 
    # input_channels=2, ##SINCOS
    # output_channels=2 ##SINCOS
    input_channels=3, ##SINCOS
    output_channels=3, ##SINCOS
).to(device)


In [ ]:
# %%time
#model = torch.compile(model)
model, scheduler, losses = train(
    model, 
    epochs=epochs,
    batch_size=batch_size, 
    lr=learning_rate, 
    model_path=model_path, 
    sch_path=sch_path,
    dataset=LatentNSynth('training'),
    verb=True,
    verb_batch=True,
    stft_transform=stft_transform
)
plt.plot(losses)

## Generacion de audio usando difusion estable:

In [ ]:
''' 
    1 Input: Imagen de ruido

    2 Modelo de difusion la transofrma en espacio latente del VAE
    
    3 Vae decodifica la imagen latente generando un espectrograma
    
    4 Transformar el espectrograma en 
'''